In [50]:
import nfl_data_py as nfl
YEARS = list(range(2018, 2025))
wk = nfl.import_weekly_data(YEARS, downcast=True)
print(wk.head())
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt


Downcasting floats.
    player_id player_name player_display_name position position_group  \
0  00-0019596     T.Brady           Tom Brady       QB             QB   
1  00-0019596     T.Brady           Tom Brady       QB             QB   
2  00-0019596     T.Brady           Tom Brady       QB             QB   
3  00-0019596     T.Brady           Tom Brady       QB             QB   
4  00-0019596     T.Brady           Tom Brady       QB             QB   

                                        headshot_url recent_team  season  \
0  https://static.www.nfl.com/image/private/f_aut...          NE    2018   
1  https://static.www.nfl.com/image/private/f_aut...          NE    2018   
2  https://static.www.nfl.com/image/private/f_aut...          NE    2018   
3  https://static.www.nfl.com/image/private/f_aut...          NE    2018   
4  https://static.www.nfl.com/image/private/f_aut...          NE    2018   

   week season_type  ... receiving_first_downs  receiving_epa  \
0     1         REG

BUILDING PPR LABEL

In [51]:
# make it RB and WR only
wk = wk.query("season_type == 'REG' and position in ['RB', 'WR']").copy()

In [52]:
#building PPR label with raw stats
if 'fumbles_lost' not in wk.columns:
    wk['fumbles_lost'] = 0

def ppr(df):
    return(
        df['receptions'].fillna(0)
        + (df['receiving_yards'].fillna(0)/10.0)
        + (df['receiving_tds'].fillna(0)*6.0)
        + (df['rushing_yards'].fillna(0)/10.0)
        + (df['rushing_yards'].fillna(0)*6.0)
        + (df['fumbles_lost'].fillna(0) * -2.0)

    )
wk['ppr_label'] = ppr(wk)



In [53]:
wk['ppr_diff'] = (wk['fantasy_points_ppr'].fillna(0) - wk['ppr_label']).abs()

print(wk[['player_display_name','position','recent_team','season','week',
          'fantasy_points_ppr','ppr_label']].head(10))
print("\nPPR diff summary:\n", wk['ppr_diff'].describe())
label = 'fantasy_points_ppr'
# Use the built-in PPR as the target
wk['label_fp'] = wk['fantasy_points_ppr'].fillna(0)


   player_display_name position recent_team  season  week  fantasy_points_ppr  \
75    Larry Fitzgerald       WR         ARI    2018     1           14.600000   
76    Larry Fitzgerald       WR         ARI    2018     2            5.800000   
77    Larry Fitzgerald       WR         ARI    2018     3            2.900000   
78    Larry Fitzgerald       WR         ARI    2018     4            5.800000   
79    Larry Fitzgerald       WR         ARI    2018     5            5.500000   
80    Larry Fitzgerald       WR         ARI    2018     6            8.900000   
81    Larry Fitzgerald       WR         ARI    2018     7           14.000000   
82    Larry Fitzgerald       WR         ARI    2018     8           26.200001   
83    Larry Fitzgerald       WR         ARI    2018    10           11.000000   
84    Larry Fitzgerald       WR         ARI    2018    11           16.299999   

    ppr_label  
75       14.6  
76        5.8  
77        2.9  
78        5.8  
79        5.5  
80        8.

Rolling and Shifting Features

In [54]:
# sort for temporal ops
wk = wk.sort_values(['player_id','season','week']).copy()
grp = wk.groupby('player_id', group_keys=False)
stats = [
    'targets','receptions','receiving_yards',
    'rushing_yards','carries',         # your table uses 'carries' (not rushing_attempts)
    'target_share'
]

for col in stats:
    if col in wk.columns:
        wk[f'{col}_lag1']  = grp[col].shift(1)
        wk[f'roll3_{col}'] = grp[col].shift(1).rolling(3, min_periods=1).mean()
        wk[f'delta_{col}'] = wk[f'{col}_lag1'] - grp[col].shift(2)



In [55]:
# 4) Quick sanity peek
print(
    wk[['player_display_name','season','week','targets','roll3_targets',
        'receptions','roll3_receptions']].head(12)
)


   player_display_name  season  week  targets  roll3_targets  receptions  \
75    Larry Fitzgerald    2018     1       10            NaN           7   
76    Larry Fitzgerald    2018     2        5      10.000000           3   
77    Larry Fitzgerald    2018     3        2       7.500000           2   
78    Larry Fitzgerald    2018     4        7       5.666667           3   
79    Larry Fitzgerald    2018     5        3       4.666667           2   
80    Larry Fitzgerald    2018     6        8       4.000000           5   
81    Larry Fitzgerald    2018     7        8       6.000000           4   
82    Larry Fitzgerald    2018     8       12       6.333333           8   
83    Larry Fitzgerald    2018    10       10       9.333333           6   
84    Larry Fitzgerald    2018    11        4      10.000000           2   
85    Larry Fitzgerald    2018    12        2       8.666667           2   
86    Larry Fitzgerald    2018    13        6       5.333333           3   

    roll3_r

Account for Defensive Matchups

In [56]:

#aggregate PPR allowed by each team for RB and WR each week
def_allowed = (
    wk.groupby(['opponent_team','season','week','position'], as_index=False)
      .agg(fp_allowed=('label_fp','sum'))  # total fantasy pts given up
      .rename(columns={'opponent_team':'defense'})
)

In [57]:
#pivot so RB and WR have their own sections
def_pivot = (
    def_allowed.pivot_table(index=['defense','season','week'],
                            columns='position', values='fp_allowed',
                            fill_value=0)
    .reset_index()
)
def_pivot.columns.name = None
def_pivot = def_pivot.rename(columns={'RB':'def_fp_allowed_RB','WR':'def_fp_allowed_WR'})

In [58]:
#create rolling averages for defenses for the past 3 weeks
gdef = def_pivot.sort_values(['defense','season','week']).groupby('defense', group_keys=False)
for col in ['def_fp_allowed_RB','def_fp_allowed_WR']:
    def_pivot[f'{col}_roll3'] = gdef[col].shift(1).rolling(3, min_periods=1).mean()

In [59]:
#merge those columns to the main week data
wk = wk.merge(
    def_pivot[['defense','season','week',
               'def_fp_allowed_RB_roll3','def_fp_allowed_WR_roll3']],
    left_on=['opponent_team','season','week'],
    right_on=['defense','season','week'],
    how='left'
).drop(columns='defense')

In [60]:
# --- consolidate duplicate columns from merges ---
import pandas as pd

def coalesce_one(df, base, how='max'):
    # collect variants like base, base_x, base_y
    cols = [c for c in df.columns if c == base or c.startswith(base + '_')]
    if not cols:
        return df
    if how == 'sum':
        df[base] = df[cols].sum(axis=1, min_count=1)
    else:  # 'max' avoids double-counting if dup columns have same value
        df[base] = df[cols].max(axis=1, skipna=True)
    # drop variants except the canonical base
    drop_cols = [c for c in cols if c != base]
    df.drop(columns=drop_cols, inplace=True, errors='ignore')
    df[base] = df[base].fillna(0)
    return df

wk = coalesce_one(wk, 'missing_targets', how='max')  # prefer max over sum to avoid doubling
wk = coalesce_one(wk, 'inj_count', how='max')


In [61]:
# 1) build the SAFE list
feature_cols = [c for c in feature_columns if c in wk.columns and not c.endswith(('_x','_y'))]

# (optional) add engineered features only if they really exist
for c in ['missing_targets', 'inj_count']:
    if c in wk.columns and c not in feature_cols:
        feature_cols.append(c)

# 2) USE feature_cols (not feature_columns) to slice X
X = wk[feature_cols].fillna(0)
y = wk['label_fp'].fillna(0)

# 3) split
train = wk['season'] < 2023
X_train, X_test = X[train], X[~train]
y_train, y_test = y[train], y[~train]


In [62]:
# --- Mission 8.2: Target Vacuum = "missing_targets" ---

# 0) Make sure we DO have roll3_targets (shifted) already
if 'roll3_targets' not in wk.columns:
    wk = wk.sort_values(['player_id','season','week']).copy()
    grp = wk.groupby('player_id', group_keys=False)
    wk['roll3_targets'] = grp['targets'].shift(1).rolling(3, min_periods=1).mean()

# 1) Load injuries
inj = nfl.import_injuries(range(2018, 2025)).copy()

# 2) Filter to players likely unavailable:
status_col = 'report_status' if 'report_status' in inj.columns else None
prac_col   = 'practice_status' if 'practice_status' in inj.columns else None

mask_out = pd.Series(False, index=inj.index)
if status_col:
    mask_out = mask_out | inj[status_col].isin(['Out', 'Doubtful'])
if prac_col:
    mask_out = mask_out | inj[prac_col].isin(['Did Not Practice'])

inj_out = inj.loc[mask_out].copy()

# 3) Use GSIS id to match injuries to weekly rows (wk.player_id is GSIS-style like '00-0033873')
#    Deduplicate to one row per player-week
keys = ['gsis_id','season','week']
inj_out = inj_out.dropna(subset=['gsis_id']) if 'gsis_id' in inj_out.columns else inj_out
inj_out = inj_out.drop_duplicates(subset=[k for k in keys if k in inj_out.columns])

# 4) Build a slim weekly usage table we can merge to (has each player's roll3_targets and team that week)
usage = wk[['player_id','season','week','recent_team','roll3_targets']].copy()

# 5) Join OUT players to their recent usage to estimate missing targets
if 'gsis_id' in inj_out.columns:
    miss = inj_out.merge(
        usage,
        left_on=['gsis_id','season','week'],
        right_on=['player_id','season','week'],
        how='left'
    )
else:
    # Fallback join by name+team if gsis_id missing (less precise)
    miss = inj_out.merge(
        wk[['player_id','player_display_name','season','week','recent_team','roll3_targets']],
        left_on=['full_name','season','week','team'],
        right_on=['player_displayname','season','week','recent_team'],
        how='left'
    )

# 6) Sum "recent targets" of OUT players per team-week  => missing_targets
vacuum = (
    miss.groupby(['recent_team','season','week'], as_index=False)['roll3_targets']
        .sum()
        .rename(columns={'roll3_targets':'missing_targets'})
)

# 7) Merge back to main table; fill NaN with 0
wk = wk.merge(vacuum, on=['recent_team','season','week'], how='left')
wk['missing_targets'] = wk['missing_targets'].fillna(0.0)

# 8) Add to feature list if not present
if 'missing_targets' not in feature_columns:
    feature_columns.append('missing_targets')


In [63]:
#if there are any missing values cus of missed games etc, we fill them with 0
X = X.fillna(0)
y = y.fillna(0)

In [64]:
#now we split the data so we train for all years before 2023, and then we will test with the newer more recent data
train = wk['season'] < 2023
X_train, X_test = X[train], X[~train]
y_train, y_test = y[train], y[~train]

print(X_train.shape, X_test.shape)


(17000, 14) (7031, 14)


In [65]:
#we use Ridge to prevent overfitting from scalars because we have so many factors
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
print(f"Baseline MAE: {mae:.2f}")



Baseline MAE: 5.01


In [66]:
compare = pd.DataFrame({
    'actual': y_test.values,
    'predicted': preds
})
print(compare.head(10))


   actual  predicted
0     1.3   5.966776
1     7.8   4.031157
2     2.5   4.164524
3     1.0   4.229758
4     1.6   5.034170
5     1.5   4.239529
6    17.4   4.699910
7     3.3   5.344331
8     0.0   4.841770
9     0.0   4.270791


In [67]:
#interpret the model
coef_df = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': model.coef_
}).sort_values(by='coefficient', ascending=False)

print("\nTop positive impact features:")
print(coef_df.head(5))

print("\nTop negative impact features:")
print(coef_df.tail(5))




Top positive impact features:
               feature  coefficient
5   roll3_target_share     9.079665
11   target_share_lag1     3.940145
6         targets_lag1     0.274397
0        roll3_targets     0.175425
10        carries_lag1     0.140394

Top negative impact features:
                    feature  coefficient
13  def_fp_allowed_WR_roll3     0.012480
7           receptions_lag1     0.004323
9        rushing_yards_lag1     0.001039
8      receiving_yards_lag1    -0.002121
12            delta_targets    -0.140956


In [113]:
coef_df.plot(kind='barh', x='feature', figsize=(8,10))
plt.show()


In [69]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Try a Random Forest model
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_preds)

print(f"\nRandom Forest MAE: {rf_mae:.2f}")



Random Forest MAE: 5.04


In [70]:
preds = model.predict(X_test)
compare = pd.DataFrame({"actual": y_test, "predicted": preds})


In [71]:
compare = compare.copy()
compare["decision"] = compare["predicted"].apply(lambda x: "START" if x >= compare["predicted"].quantile(0.70) else "SIT")

print(compare.head(15))


     actual  predicted decision
724     1.3   5.966776      SIT
725     7.8   4.031157      SIT
726     2.5   4.164524      SIT
727     1.0   4.229758      SIT
728     1.6   5.034170      SIT
729     1.5   4.239529      SIT
730    17.4   4.699910      SIT
731     3.3   5.344331      SIT
859     0.0   4.841770      SIT
860     0.0   4.270791      SIT
861     2.2   3.726918      SIT
862     2.8   3.749776      SIT
863     0.0   5.010664      SIT
864     2.0   4.674545      SIT
865     8.5   4.436417      SIT


In [72]:
from sklearn.metrics import accuracy_score

compare["actual_start"] = compare["actual"].apply(lambda x: "START" if x >= compare["actual"].quantile(0.70) else "SIT")

print("Start/Sit accuracy:", accuracy_score(compare["actual_start"], compare["decision"]))


Start/Sit accuracy: 0.7506755795761627


In [73]:
def recommend_starters(roster_df, model, wk_df, n_starters=5):
    # Filter wk_df for only the player IDs on your roster
    merged = wk_df[wk_df['player_display_name'].isin(roster_df['player_display_name'])].copy()

    # Use the same feature columns you trained on
    merged = merged[feature_cols]

    # Predict fantasy points
    merged['projected'] = model.predict(merged)

    return merged.sort_values("projected", ascending=False).head(n_starters)


In [74]:
# --- LIVE WEEK PREDICTION: BUILD wk_live WITH SAME FEATURES AS TRAINING ---

CURRENT_SEASON = 2024  # last completed season

wk_live = nfl.import_weekly_data([CURRENT_SEASON], downcast=True).sort_values(
    ['player_id','season','week']
)

g = wk_live.groupby('player_id', group_keys=False)

wk_live['roll3_targets']       = g['targets'].shift(1).rolling(3, min_periods=1).mean()
wk_live['roll3_target_share']  = g['target_share'].shift(1).rolling(3, min_periods=1).mean()
wk_live['targets_lag1']        = g['targets'].shift(1)
wk_live['target_share_lag1']   = g['target_share'].shift(1)


Downcasting floats.


In [75]:
# --- DEFENSE ALLOWED rolling features (must match training logic) ---

def_allowed = (
    wk_live.groupby(['opponent_team','season','week','position'], as_index=False)
           .agg(fp_allowed=('fantasy_points_ppr','sum'))
           .rename(columns={'opponent_team': 'defense'})
)

def_pivot = def_allowed.pivot_table(
    index=['defense','season','week'],
    columns='position',
    values='fp_allowed',
    fill_value=0
).reset_index()

def_pivot.columns.name = None
def_pivot = def_pivot.rename(columns={'RB':'def_fp_allowed_RB', 'WR':'def_fp_allowed_WR'})

gdef = def_pivot.sort_values(['defense','season','week']).groupby('defense', group_keys=False)
for col in ['def_fp_allowed_RB','def_fp_allowed_WR']:
    def_pivot[f'{col}_roll3'] = gdef[col].shift(1).rolling(3, min_periods=1).mean()

wk_live = wk_live.merge(
    def_pivot[['defense','season','week','def_fp_allowed_RB_roll3','def_fp_allowed_WR_roll3']],
    left_on=['opponent_team','season','week'],
    right_on=['defense','season','week'],
    how='left'
).drop(columns=['defense'])


In [76]:
# --- ALIGN COLUMNS & PREDICT ---

LAST_WEEK = int(wk_live['week'].max())
wk_current = wk_live[wk_live['week'] == LAST_WEEK].copy()

expected = list(rf.feature_names_in_)   # EXACT columns from training

# create missing cols as 0 for live data
for c in expected:
    if c not in wk_current.columns:
        wk_current[c] = 0.0

# reorder to match training order
X_live = wk_current[expected].fillna(0)

wk_current['projected_ppr'] = rf.predict(X_live)


In [79]:
print("wk_current columns:", wk_current.columns.tolist()[:15], "...")
print("Sample names:", wk_current["player_display_name"].dropna().unique()[:15])


wk_current columns: ['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'recent_team', 'season', 'week', 'season_type', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds'] ...
Sample names: ['Travis Kelce' 'DeAndre Hopkins' 'Samaje Perine' 'JuJu Smith-Schuster'
 'Patrick Mahomes' 'Kareem Hunt' 'Dallas Goedert' 'Justin Watson'
 'Saquon Barkley' 'Marquise Brown' 'A.J. Brown' 'Jalen Hurts' 'Noah Gray'
 'DeVonta Smith' 'Kenneth Gainwell']


In [93]:
import pandas as pd, re

CURRENT_SEASON = 2024  # keep as your live season

# 1) Load the season and keep REG + RB/WR only
wk_live = (
    nfl.import_weekly_data([CURRENT_SEASON], downcast=True)
      .sort_values(['player_id','season','week'])
      .query("season_type == 'REG' and position in ['RB','WR']")
      .copy()
)

# 2) Recreate the SAME features as training
g = wk_live.groupby('player_id', group_keys=False)
wk_live['roll3_targets']        = g['targets'].shift(1).rolling(3, min_periods=1).mean()
wk_live['roll3_receptions']     = g['receptions'].shift(1).rolling(3, min_periods=1).mean()
wk_live['roll3_receiving_yards']= g['receiving_yards'].shift(1).rolling(3, min_periods=1).mean()
wk_live['roll3_rushing_yards']  = g['rushing_yards'].shift(1).rolling(3, min_periods=1).mean()
wk_live['roll3_carries']        = g['carries'].shift(1).rolling(3, min_periods=1).mean()
wk_live['roll3_target_share']   = g['target_share'].shift(1).rolling(3, min_periods=1).mean()

wk_live['targets_lag1']         = g['targets'].shift(1)
wk_live['receptions_lag1']      = g['receptions'].shift(1)
wk_live['receiving_yards_lag1'] = g['receiving_yards'].shift(1)
wk_live['rushing_yards_lag1']   = g['rushing_yards'].shift(1)
wk_live['carries_lag1']         = g['carries'].shift(1)
wk_live['target_share_lag1']    = g['target_share'].shift(1)

wk_live['delta_targets']        = wk_live['targets_lag1'] - g['targets'].shift(2)
wk_live['delta_receptions']     = wk_live['receptions_lag1'] - g['receptions'].shift(2)

# 3) Defensive allowance (same as training)
def_allowed = (
    wk_live.groupby(['opponent_team','season','week','position'], as_index=False)
           .agg(fp_allowed=('fantasy_points_ppr','sum'))
           .rename(columns={'opponent_team':'defense'})
)
def_pivot = (
    def_allowed.pivot_table(index=['defense','season','week'],
                            columns='position', values='fp_allowed',
                            fill_value=0)
    .reset_index()
)
def_pivot.columns.name = None
def_pivot = def_pivot.rename(columns={'RB':'def_fp_allowed_RB','WR':'def_fp_allowed_WR'})
gdef = def_pivot.sort_values(['defense','season','week']).groupby('defense', group_keys=False)
for col in ['def_fp_allowed_RB','def_fp_allowed_WR']:
    def_pivot[f'{col}_roll3'] = gdef[col].shift(1).rolling(3, min_periods=1).mean()

wk_live = wk_live.merge(
    def_pivot[['defense','season','week','def_fp_allowed_RB_roll3','def_fp_allowed_WR_roll3']],
    left_on=['opponent_team','season','week'],
    right_on=['defense','season','week'],
    how='left'
).drop(columns='defense')

# 4) Use the last REG week
LAST_REG_WEEK = int(wk_live['week'].max())
wk_current = wk_live[wk_live['week'] == LAST_REG_WEEK].copy()
print("wk_current REG week:", LAST_REG_WEEK, wk_current.shape)


Downcasting floats.
wk_current REG week: 18 (199, 69)


In [112]:
# === START/SIT: force current week, fix names, and backfill missing players ===
import pandas as pd, re

# ---- 0) Helper: normalize names + aliases for common variants
def norm(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"[.\-']", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

name_alias = {
    "devonta smith": "DeVonta Smith",

    "amon ra st brown": "Amon-Ra St. Brown",
    "christian mccaffrey": "Christian McCaffrey",
}

# ---- 1) Your roster (RB/WR only) — edit this list as needed
my_team = pd.DataFrame({
    "player_display_name": [
        "Christian McCaffrey",
        "Amon-Ra St. Brown",
        "Jordan Addison",
        "DeVonta Smith",
    ]
})
# normalize to canonical display names, then build merge key
my_team["player_display_name"] = my_team["player_display_name"].map(lambda x: name_alias.get(norm(x), x))
my_team["merge_name"] = my_team["player_display_name"].map(norm)
want = set(my_team["merge_name"])

# ---- 2) Build live season table for REG season RB/WR with SAME FEATURES as training
CURRENT_SEASON = 2024  # change if you want a different season

wk_live = (
    nfl.import_weekly_data([CURRENT_SEASON], downcast=True)
      .sort_values(['player_id','season','week'])
      .query("season_type == 'REG' and position in ['RB','WR']")
      .copy()
)

# rolling/lag features (identical to training)
g = wk_live.groupby('player_id', group_keys=False)
wk_live['roll3_targets']         = g['targets'].shift(1).rolling(3, min_periods=1).mean()
wk_live['roll3_receptions']      = g['receptions'].shift(1).rolling(3, min_periods=1).mean()
wk_live['roll3_receiving_yards'] = g['receiving_yards'].shift(1).rolling(3, min_periods=1).mean()
wk_live['roll3_rushing_yards']   = g['rushing_yards'].shift(1).rolling(3, min_periods=1).mean()
wk_live['roll3_carries']         = g['carries'].shift(1).rolling(3, min_periods=1).mean()
wk_live['roll3_target_share']    = g['target_share'].shift(1).rolling(3, min_periods=1).mean()

wk_live['targets_lag1']          = g['targets'].shift(1)
wk_live['receptions_lag1']       = g['receptions'].shift(1)
wk_live['receiving_yards_lag1']  = g['receiving_yards'].shift(1)
wk_live['rushing_yards_lag1']    = g['rushing_yards'].shift(1)
wk_live['carries_lag1']          = g['carries'].shift(1)
wk_live['target_share_lag1']     = g['target_share'].shift(1)

wk_live['delta_targets']         = wk_live['targets_lag1']     - g['targets'].shift(2)
wk_live['delta_receptions']      = wk_live['receptions_lag1']  - g['receptions'].shift(2)

# defensive roll-3 allowance (same logic as training)
def_allowed = (
    wk_live.groupby(['opponent_team','season','week','position'], as_index=False)
           .agg(fp_allowed=('fantasy_points_ppr','sum'))
           .rename(columns={'opponent_team':'defense'})
)
def_pivot = (
    def_allowed.pivot_table(index=['defense','season','week'],
                            columns='position', values='fp_allowed', fill_value=0)
    .reset_index()
)
def_pivot.columns.name = None
def_pivot = def_pivot.rename(columns={'RB':'def_fp_allowed_RB','WR':'def_fp_allowed_WR'})
gdef = def_pivot.sort_values(['defense','season','week']).groupby('defense', group_keys=False)
for col in ['def_fp_allowed_RB','def_fp_allowed_WR']:
    def_pivot[f'{col}_roll3'] = gdef[col].shift(1).rolling(3, min_periods=1).mean()

wk_live = wk_live.merge(
    def_pivot[['defense','season','week','def_fp_allowed_RB_roll3','def_fp_allowed_WR_roll3']],
    left_on=['opponent_team','season','week'],
    right_on=['defense','season','week'],
    how='left'
).drop(columns='defense')

# ---- 3) Force CURRENT WEEK from data feed
CURRENT_WEEK = int(wk_live['week'].max())
wk_current = wk_live[wk_live['week'] == CURRENT_WEEK].copy()
wk_current['merge_name'] = wk_current['player_display_name'].map(norm)
print(f"Using CURRENT WEEK: {CURRENT_WEEK}  | wk_current rows: {len(wk_current)}")

# ---- 4) Align to model columns & predict for CURRENT WEEK
expected = list(rf.feature_names_in_)
for c in expected:
    if c not in wk_current.columns:
        wk_current[c] = 0.0
X_live = wk_current[expected].fillna(0)
wk_current['projected_ppr'] = rf.predict(X_live)

# ---- 5) Merge roster; for any missing players, backfill using most recent REG appearance
roster_proj = wk_current.merge(my_team, on="merge_name", how="inner")

missing = want - set(roster_proj["merge_name"])
if missing:
    # last regular-season appearance for each missing player
    wk_live["merge_name"] = wk_live["player_display_name"].map(norm)
    last_rows = (
        wk_live[wk_live["merge_name"].isin(missing)]
        .sort_values(['merge_name','week'])
        .groupby('merge_name', as_index=False)
        .tail(1)
        .copy()
    )
    # predict for backfilled rows
    for c in expected:
        if c not in last_rows.columns:
            last_rows[c] = 0.0
    X_last = last_rows[expected].fillna(0)
    last_rows['projected_ppr'] = rf.predict(X_last)
    last_rows['note'] = "used most recent REG appearance (week " + last_rows['week'].astype(int).astype(str) + ")"
    # append
    keep_cols = ['merge_name','player_display_name','position','recent_team','opponent_team','projected_ppr','note']
    roster_proj = pd.concat([roster_proj, last_rows[keep_cols]], ignore_index=True)

# ---- 6) Display results (and who’s still missing, if any)
final_cols = [c for c in ["player_display_name","position","recent_team","opponent_team","projected_ppr","note"] if c in roster_proj.columns]
print(roster_proj[final_cols].sort_values("projected_ppr", ascending=False))

still_missing = want - set(roster_proj["merge_name"])
print("Still not found:", still_missing)

# Optional: export
# roster_proj[final_cols].round(2).to_csv("start_sit_projection.csv", index=False)
# print("Saved -> start_sit_projection.csv")


Downcasting floats.
Using CURRENT WEEK: 18  | wk_current rows: 199
   player_display_name position recent_team opponent_team  projected_ppr  \
3        DeVonta Smith       WR         PHI           DAL      18.121801   
2  Christian McCaffrey       RB          SF           BUF      16.884887   
0                  NaN       WR         DET           MIN      16.465433   
1                  NaN       WR         MIN           DET      11.892273   

                                        note  
3  used most recent REG appearance (week 17)  
2  used most recent REG appearance (week 13)  
0                                        NaN  
1                                        NaN  
Still not found: set()


Using CURRENT WEEK: 18
